# Observation 0 - Amount violin plots by age bucket

This notebook renders violin plots of `Opportunity Amount USD` across fine age buckets,
plus controlled views by key categorical variables (especially company-size fields).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
pd.options.display.max_rows = 200
pd.options.display.max_columns = 200


In [2]:
base_dir = Path.cwd().resolve()
candidate_roots = [base_dir, base_dir.parent, base_dir.parent.parent]
input_path = None
project_root = None
for root in candidate_roots:
    candidate = root / "observation_0" / "observation0_full_with_maturity.csv"
    if candidate.exists():
        input_path = candidate
        project_root = root
        break

if input_path is None:
    raise FileNotFoundError("Could not locate observation0_full_with_maturity.csv from current working directory context")

obs_dir = input_path.parent
output_dir = obs_dir / "02_amount_violinplots_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(input_path)
print(f"Loaded rows: {len(df):,}")
print(f"Input: {input_path}")
print(f"Output dir: {output_dir}")
print(f"Project root: {project_root}")

Loaded rows: 78,025
Input: /mnt/HC_Volume_104595655/home/goviedb/repos/mib/cars_analysis/observation_0/observation0_full_with_maturity.csv
Output dir: /mnt/HC_Volume_104595655/home/goviedb/repos/mib/cars_analysis/observation_0/02_amount_violinplots_outputs
Project root: /mnt/HC_Volume_104595655/home/goviedb/repos/mib/cars_analysis


In [3]:
fine_bins = [0, 15, 30, 45, 60, 75, 90, 120, np.inf]
fine_labels = ["0-15", "16-30", "31-45", "46-60", "61-75", "76-90", "91-120", "121+"]

df["age_bucket_fine"] = pd.cut(
    df["Elapsed Days In Sales Stage"],
    bins=fine_bins,
    labels=fine_labels,
    right=True,
    include_lowest=True,
)

df["amount_usd"] = pd.to_numeric(df["Opportunity Amount USD"], errors="coerce")
df = df.dropna(subset=["amount_usd", "age_bucket_fine"]).copy()
df["amount_usd"] = df["amount_usd"].clip(lower=0)
df["amount_usd_log1p"] = np.log1p(df["amount_usd"])

for col in ["Client Size By Revenue (USD)", "Client Size By Employee Count", "Region", "Route To Market", "Opportunity Result"]:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown").astype(str)

bucket_counts = df["age_bucket_fine"].value_counts().reindex(fine_labels, fill_value=0)
bucket_counts

age_bucket_fine
0-15      11752
16-30     18370
31-45     13018
46-60     10814
61-75     11744
76-90     10649
91-120     1640
121+         38
Name: count, dtype: int64

In [4]:
def top_n_with_other(series: pd.Series, top_n: int = 6) -> pd.Series:
    top_values = series.value_counts(dropna=False).head(top_n).index
    return series.where(series.isin(top_values), "Other")

def save_violin_plot(data: pd.DataFrame, x: str, y: str, title: str, out_name: str, order=None):
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.violinplot(data=data, x=x, y=y, order=order, inner="quartile", cut=0, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Age bucket")
    ax.set_ylabel("Opportunity Amount USD (log1p)")
    fig.tight_layout()
    out_path = output_dir / out_name
    fig.savefig(out_path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    return out_path

def save_controlled_violin_grid(data: pd.DataFrame, control_col: str, out_name: str, top_n: int = 6, order=None):
    local = data.copy()
    local["control_group"] = top_n_with_other(local[control_col], top_n=top_n)
    groups = local["control_group"].value_counts().index.tolist()

    n_groups = len(groups)
    ncols = 2
    nrows = int(np.ceil(n_groups / ncols))
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(16, 4.3 * nrows), sharey=True)
    axes = np.atleast_1d(axes).ravel()

    for idx, grp in enumerate(groups):
        ax = axes[idx]
        subset = local[local["control_group"] == grp]
        sns.violinplot(data=subset, x="age_bucket_fine", y="amount_usd_log1p", order=order, inner="quartile", cut=0, ax=ax)
        ax.set_title(f"{control_col}: {grp} (n={len(subset):,})")
        ax.set_xlabel("Age bucket")
        if idx % ncols == 0:
            ax.set_ylabel("Opportunity Amount USD (log1p)")
        else:
            ax.set_ylabel("")

    for j in range(n_groups, len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(f"Amount distribution by age bucket, controlled by {control_col}", y=1.01)
    fig.tight_layout()
    out_path = output_dir / out_name
    fig.savefig(out_path, dpi=160, bbox_inches="tight")
    plt.close(fig)

    summary = (
        local.groupby(["control_group", "age_bucket_fine"], observed=False)["amount_usd"]
        .agg(n_rows="size", mean_amount="mean", median_amount="median", std_amount="std")
        .reset_index()
    )
    return out_path, summary


In [5]:
plot_paths = []
summary_paths = []

overall_plot = save_violin_plot(
    data=df,
    x="age_bucket_fine",
    y="amount_usd_log1p",
    title="Opportunity Amount (log1p) by fine age bucket",
    out_name="violin_amount_log1p_by_age_bucket_overall.png",
    order=fine_labels,
)
plot_paths.append(overall_plot)

control_vars = [
    "Client Size By Revenue (USD)",
    "Client Size By Employee Count",
    "Region",
    "Route To Market",
    "Opportunity Result",
]

for control_col in control_vars:
    if control_col not in df.columns:
        continue
    safe_name = control_col.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")
    plot_path, summary_df = save_controlled_violin_grid(
        data=df,
        control_col=control_col,
        out_name=f"violin_amount_log1p_by_age_bucket_control_{safe_name}.png",
        top_n=6,
        order=fine_labels,
    )
    summary_path = output_dir / f"summary_amount_by_age_bucket_control_{safe_name}.csv"
    summary_df.to_csv(summary_path, index=False)
    plot_paths.append(plot_path)
    summary_paths.append(summary_path)

manifest = pd.DataFrame({
    "plot_path": [str(p.relative_to(project_root)) for p in plot_paths],
})
manifest_path = output_dir / "plot_manifest.csv"
manifest.to_csv(manifest_path, index=False)

summary_manifest = pd.DataFrame({
    "summary_path": [str(p.relative_to(project_root)) for p in summary_paths],
})
summary_manifest_path = output_dir / "summary_manifest.csv"
summary_manifest.to_csv(summary_manifest_path, index=False)

print(f"Saved {len(plot_paths)} plot files")
print(f"Saved {len(summary_paths)} summary files")
print(f"Plot manifest: {manifest_path}")
print(f"Summary manifest: {summary_manifest_path}")

Saved 6 plot files
Saved 5 summary files
Plot manifest: /mnt/HC_Volume_104595655/home/goviedb/repos/mib/cars_analysis/observation_0/02_amount_violinplots_outputs/plot_manifest.csv
Summary manifest: /mnt/HC_Volume_104595655/home/goviedb/repos/mib/cars_analysis/observation_0/02_amount_violinplots_outputs/summary_manifest.csv


In [6]:
manifest

,plot_path
0,observation_0/02_amount_violinplots_outputs/vi...
1,observation_0/02_amount_violinplots_outputs/vi...
2,observation_0/02_amount_violinplots_outputs/vi...
3,observation_0/02_amount_violinplots_outputs/vi...
4,observation_0/02_amount_violinplots_outputs/vi...
5,observation_0/02_amount_violinplots_outputs/vi...


In [7]:
summary_manifest = pd.read_csv(output_dir / "summary_manifest.csv")
summary_manifest.head(10)

,summary_path
0,observation_0/02_amount_violinplots_outputs/su...
1,observation_0/02_amount_violinplots_outputs/su...
2,observation_0/02_amount_violinplots_outputs/su...
3,observation_0/02_amount_violinplots_outputs/su...
4,observation_0/02_amount_violinplots_outputs/su...
